# Reproducing Table 5 of the NRP-Exp Paper

This notebook reproduces the illustrative workflow reported in Table 5 of the paper.

Configuration:

- Instance: `Concise_Case_11`
- Variant: `Excl(r7)`
- Objectives:
  - Satisfaction (maximize)
  - Effort (minimize)
- Constraint:
  - Effort ≤ 28
- Solver:
  - Brute Force
- Parameter:
  - max_evaluations = 30000

The notebook executes exactly the workflow reported in the paper and computes the corresponding quality indicators.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Users\imagu\OneDrive\Escritorio\NRPExperimentFramework


## Import Framework Components

Import the components required to reproduce the workflow.

In [2]:
import json
import time
import pandas as pd

from src.domain.core.loader import GenericJSONLoader
from src.domain.core.pipeline import PreprocessingEngine
from src.domain.plugins.nrp import NRPPlugin

from src.solving.algorithms.brute_force import BruteForceSolver

from src.solving.core.problem import (
    OptimizationProblem,
    ConstraintSpec,
    ThresholdOperator,
)

## Load the Original Instance

Load the `Concise_Case_11` instance distributed with NRP-Exp.

In [3]:
INSTANCE_PATH = (
    PROJECT_ROOT
    / "examples"
    / "instances"
    / "paper"
    / "Concise_Case_11.json"
)

raw_json = INSTANCE_PATH.read_text(
    encoding="utf-8"
)

base_model = GenericJSONLoader.load(
    raw_json,
    NRPPlugin,
)

print(base_model.name)

Concise_Case_11


## Execute Preprocessing

Apply aggregation, coupling resolution, value-dependency processing, and exclusion branching.

In [4]:
processed_models = PreprocessingEngine.process(
    initial_model=base_model,
    pipeline_options={}
)

print(
    "Generated variants:",
    len(processed_models)
)

Generated variants: 2


## Inspect Generated Variants

The paper uses the variant corresponding to `Excl(r7)`.

In [5]:
for idx, model in enumerate(processed_models):
    print(
        idx,
        getattr(
            model,
            "branch_name",
            f"Variant {idx}"
        ),
        len(model.items)
    )

0 Excl(r7) 10
1 Excl(r9) 8


## Select the Variant Reported in Table 5

In [6]:
SELECTED_VARIANT_NAME = "Excl(r7)"

matching_models = [
    candidate
    for candidate in processed_models
    if getattr(
        candidate,
        "branch_name",
        "",
    ) == SELECTED_VARIANT_NAME
]


if len(matching_models) != 1:
    raise ValueError(
        f"Expected one variant named "
        f"{SELECTED_VARIANT_NAME!r}, "
        f"but found {len(matching_models)}."
    )

model = matching_models[0]


print(
    getattr(
        model,
        "branch_name",
        "Variant"
    )
)

Excl(r7)


## Configure Objectives and Constraints

Table 5 uses:

- Satisfaction (max)
- Effort (min)
- Effort ≤ 28

In [7]:
OBJECTIVES = {
    "satisfaction": "max",
    "effort": "min",
}

constraints = {
    "effort": ConstraintSpec(
        attribute="effort",
        operator=ThresholdOperator.LESS_EQUAL,
        value=28.0,
    )
}

problem = OptimizationProblem(
    model=model,
    objectives=OBJECTIVES,
    constraints=constraints,
)

## Execute  Brute Force

Configuration reported in Table 5:

- max_evaluations = 30000

In [8]:
solver = BruteForceSolver(
    max_evaluations=30000
)

start = time.perf_counter()

pareto_front = solver.solve(problem)

elapsed = (
    time.perf_counter() - start
)

pareto_front.set_objective_directions(
    problem.get_objective_directions()
)

print(
    "Execution time:",
    elapsed
)

print(
    "Solutions:",
    len(pareto_front.solutions)
)

Execution time: 0.028671599982772022
Solutions: 21


## Inspect the Returned Front

In [9]:
pareto_df = pareto_front.to_dataframe()

pareto_df.head(20)

,id,selected_ids,selected_ids_str,is_feasible,satisfaction,effort,dissatisfaction,risk
0,BF_2,"[r1, r10, r11, r2, r3, r4_r5, r6, r9]","r1, r10, r11, r2, r3, r4_r5, r6, r9",True,28.714286,26.2,18.428571,20.000000
1,BF_4,"[r1, r10, r2, r3, r4_r5, r6, r9]","r1, r10, r2, r3, r4_r5, r6, r9",True,27.000000,22.0,17.142857,16.666667
2,BF_5,"[r1, r11, r2, r3, r4_r5, r6, r9]","r1, r11, r2, r3, r4_r5, r6, r9",True,28.428571,24.2,15.714286,18.666667
3,BF_6,"[r1, r2, r3, r4_r5, r6, r9]","r1, r2, r3, r4_r5, r6, r9",True,26.714286,20.0,14.428571,15.333333
4,BF_8,"[r1, r2, r3, r4_r5, r6]","r1, r2, r3, r4_r5, r6",True,24.714286,17.0,12.285714,12.333333
5,BF_26,"[r1, r11, r2, r3, r6, r9]","r1, r11, r2, r3, r6, r9",True,25.571429,19.2,9.571429,14.666667
6,BF_27,"[r1, r2, r3, r6, r9]","r1, r2, r3, r6, r9",True,23.857143,15.0,8.285714,11.333333
7,BF_45,"[r1, r10, r11, r3, r6, r8, r9]","r1, r10, r11, r3, r6, r8, r9",True,28.857143,27.2,16.714286,15.666667
8,BF_47,"[r1, r10, r3, r6, r8, r9]","r1, r10, r3, r6, r8, r9",True,27.142857,23.0,15.428571,12.333333
9,BF_48,"[r1, r11, r3, r6, r8, r9]","r1, r11, r3, r6, r8, r9",True,28.571429,25.2,14.000000,14.333333


## Compute Quality Indicators

Compute the indicators reported in Table 5.

In [12]:
objective_vectors = {
    (
        round(
            float(
                solution.objectives["satisfaction"]
            ),
            6,
        ),
        round(
            float(
                solution.objectives["effort"]
            ),
            6,
        ),
    )
    for solution
    in pareto_front.solutions
}

print(
    "Plans:",
    len(pareto_front.solutions)
)

print(
    "Objective vectors:",
    len(objective_vectors)
)


metrics = pareto_front.compute_metrics(
    objective_names=[
        "satisfaction",
        "effort",
    ],
    directions={
        "satisfaction": "max",
        "effort": "min",
    }
)

metrics

Plans: 21
Objective vectors: 21


{'Solutions': 21,
 'Hypervolume': 695.6605714285714,
 'Spacing': 0.7498251920352749,
 'Objective-space extent': 39.65570190877411,
 'Spread': 39.65570190877411}

## Table 5 Summary

Generate a summary directly comparable to the values reported in the paper.

In [13]:
summary = {
    "Instance": "Concise_Case_11",
    "Variant": getattr(
        model,
        "branch_name",
        "Excl(r7)"
    ),
    "Constraint": "Effort <= 28",
    "Solver": "Brute Force",
    "max_evaluations": 30000,
    "Solutions": len(
        pareto_front.solutions
    ),
    "Objective vectors": len(
        objective_vectors
    ),
    **metrics,
}

pd.Series(summary)

Instance                  Concise_Case_11
Variant                          Excl(r7)
Constraint                   Effort <= 28
Solver                        Brute Force
max_evaluations                     30000
Solutions                              21
Objective vectors                      21
Hypervolume                    695.660571
Spacing                          0.749825
Objective-space extent          39.655702
Spread                          39.655702
dtype: object

## Reproduction Completed

This notebook reproduces the workflow reported in Table 5:

- Concise_Case_11
- Excl(r7)
- Satisfaction (max)
- Effort (min)
- Effort ≤ 28
- Brute Force
- max_evaluations = 30000

The generated outputs can be compared directly against the values reported in the manuscript.